## Required Libraries

In [52]:
!pip install xgboost

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## **Dataset**

In [54]:
file = '/home/jhord/code/AlexisKiehn/etrace/cleaned_final_dataset.csv'
df = pd.read_csv(file)
df.head()

,geo,year,pop,employment_rate,gdp_capita,NUTS_NAME,area_km2,pop_dens,pct_Dfb,pct_Dfc,...,pct_Cfc,pct_BWh,pct_Af,pct_Am,pct_Aw,pct_Cwa,pct_Cwb,pct_Csc,pct_Dsa,nights_spent
0,AT11,2000,276226.0,70.8,17700.0,Burgenland,3965.379817,69.659405,0.598559,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2099045.0
1,AT12,2000,1535083.0,71.8,21900.0,Niederösterreich,19196.454858,79.967005,0.801834,0.018312,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5170345.0
2,AT13,2000,1548537.0,70.2,36400.0,Wien,414.060183,3739.883870,0.305616,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7697066.0
3,AT21,2000,560696.0,66.9,22400.0,Kärnten,9543.157976,58.753717,0.655037,0.252566,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10282335.0
4,AT22,2000,1182930.0,67.4,23200.0,Steiermark,16413.667048,72.069818,0.704175,0.260468,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7416045.0


In [55]:
# Ordering by geo and year
df = df.sort_values(by=['geo', 'year'], ascending=True)

# Handling missing data through interpolation
df['gdp_capita'] = df['gdp_capita'].interpolate(method='linear', limit_direction='forward')
df['pop'] = df['pop'].interpolate(method='linear', limit_direction='forward')
df['employment_rate'] = df['employment_rate'].interpolate(method='linear', limit_direction='forward')

counts = df["geo"].value_counts()

print('# unique geo values:', len(df['geo'].unique()))
print(counts)

# unique geo values: 282
geo
SE33    21
SE32    21
SE31    21
SE23    21
SE22    21
        ..
NO0A     1
RS22     1
RS21     1
RS11     1
RS12     1
Name: count, Length: 282, dtype: int64


In [56]:
# Filter for only completed data
df_clean = df.groupby("geo").filter(lambda g: len(g) == counts.max()) # Select NUTS2 with completed data

counts_2 = df_clean["geo"].value_counts().sort_index()

print('# unique geo values:', len(df_clean['geo'].unique()))
print(counts_2)

# unique geo values: 127
geo
AT11    21
AT12    21
AT13    21
AT21    21
AT22    21
        ..
SE22    21
SE23    21
SE31    21
SE32    21
SE33    21
Name: count, Length: 127, dtype: int64


In [62]:
df_clean.head()

,geo,year,pop,employment_rate,gdp_capita,NUTS_NAME,area_km2,pop_dens,pct_Dfb,pct_Dfc,...,pct_Am,pct_Aw,pct_Cwa,pct_Cwb,pct_Csc,pct_Dsa,nights_spent,lag1,lag2,diff1
82,ES11,2000,2702471.0,59.8,12400.0,Galicia,29572.787517,91.383709,0.0,0.000011,...,0.0,0.0,0.0,0.0,0.0,0.0,6713632.0,NaN,NaN,NaN
234,ES11,2001,2698025.0,60.3,13400.0,Galicia,29572.787517,91.233368,0.0,0.000009,...,0.0,0.0,0.0,0.0,0.0,0.0,6765876.0,6713632.0,NaN,52244.0
418,ES11,2002,2696809.0,60.6,14200.0,Galicia,29572.787517,91.192249,0.0,0.000007,...,0.0,0.0,0.0,0.0,0.0,0.0,7522946.0,6765876.0,6713632.0,757070.0
595,ES11,2003,2705159.0,62.6,15200.0,Galicia,29572.787517,91.474603,0.0,0.000005,...,0.0,0.0,0.0,0.0,0.0,0.0,7640816.0,7522946.0,6765876.0,117870.0
777,ES11,2004,2711586.0,62.5,16300.0,Galicia,29572.787517,91.691931,0.0,0.000002,...,0.0,0.0,0.0,0.0,0.0,0.0,9191098.0,7640816.0,7522946.0,1550282.0


In [63]:
# Select only Spain for trial. Delete then
df_clean = df_clean[df_clean["geo"].str.match(r"^ES[12]")]
print('# unique geo values:', len(df_clean['geo'].unique()))
df_clean.head()

# unique geo values: 7


,geo,year,pop,employment_rate,gdp_capita,NUTS_NAME,area_km2,pop_dens,pct_Dfb,pct_Dfc,...,pct_Am,pct_Aw,pct_Cwa,pct_Cwb,pct_Csc,pct_Dsa,nights_spent,lag1,lag2,diff1
82,ES11,2000,2702471.0,59.8,12400.0,Galicia,29572.787517,91.383709,0.0,0.000011,...,0.0,0.0,0.0,0.0,0.0,0.0,6713632.0,NaN,NaN,NaN
234,ES11,2001,2698025.0,60.3,13400.0,Galicia,29572.787517,91.233368,0.0,0.000009,...,0.0,0.0,0.0,0.0,0.0,0.0,6765876.0,6713632.0,NaN,52244.0
418,ES11,2002,2696809.0,60.6,14200.0,Galicia,29572.787517,91.192249,0.0,0.000007,...,0.0,0.0,0.0,0.0,0.0,0.0,7522946.0,6765876.0,6713632.0,757070.0
595,ES11,2003,2705159.0,62.6,15200.0,Galicia,29572.787517,91.474603,0.0,0.000005,...,0.0,0.0,0.0,0.0,0.0,0.0,7640816.0,7522946.0,6765876.0,117870.0
777,ES11,2004,2711586.0,62.5,16300.0,Galicia,29572.787517,91.691931,0.0,0.000002,...,0.0,0.0,0.0,0.0,0.0,0.0,9191098.0,7640816.0,7522946.0,1550282.0


In [ ]:
cols_climate = [c for c in df_clean.columns if c.startswith("pct_")]
# cols_climate = ['pct_BWh', 'pct_BSh', 'pct_Dfb']
cols_econ = ["year", "pop", "employment_rate", "gdp_capita", "pop_dens"]
cols_cat = ['geo']

features = cols_cat + cols_econ + cols_climate
target = 'nights_spent'

print("Clima:", cols_climate)
print("Econ:", cols_econ)
print("Categorías:", cols_cat)

Clima: ['pct_Dfb', 'pct_Dfc', 'pct_ET', 'pct_EF', 'pct_Cfb', 'pct_BSk', 'pct_Csa', 'pct_Csb', 'pct_Cfa', 'pct_Dfa', 'pct_Dsb', 'pct_Dsc', 'pct_BSh', 'pct_BWk', 'pct_Cfc', 'pct_BWh', 'pct_Af', 'pct_Am', 'pct_Aw', 'pct_Cwa', 'pct_Cwb', 'pct_Csc', 'pct_Dsa']
Econ: ['year', 'pop', 'employment_rate', 'gdp_capita', 'pop_dens']
Categorías: ['geo']


In [68]:
# Could replace before cells

#cols_climate = [c for c in df_clean.columns if c.startswith("pct_")]
#cols_econ = [c for c in df_clean.select_dtypes(include=['int64', 'float64']).columns
#             if c not in cols_climate]
#cols_cat = list(df_clean.select_dtypes(include=['object', 'category']).columns)

#print("Clima:", cols_climate)
#print("Econ:", cols_econ)
#print("Categorías:", cols_cat)

In [69]:
X = df_clean[features]
y = df_clean[target]

X.shape, y.shape

((147, 29), (147,))

In [70]:
# Categorical and number columns
cat_cols = cols_cat
num_cols = cols_econ + cols_climate

In [ ]:
# Preprocessing
from sklearn.preprocessing import OneHotEncoder, RobustScaler

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols), # Review this part
    ("num", RobustScaler(), num_cols)  # Scaling
])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

results_dict = {}

for region, group in df_clean.groupby("geo"):

    # Split temporal using int year
    train = group[group["year"] <= 2018]
    test  = group[group["year"] >  2018]

    # Validate that there is sufficient data
    if len(train) < 2 or len(test) < 2:
        print(f"Geo {region} has little data and is omitted")
        continue


    # Model XGB
    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    # Pipeline
    pipeline = Pipeline([
        ("prep", preprocessor),
        ("xgb", model)
    ])

    # Split data for train and test
    X_train = train[features]
    y_train = train[target]
    X_test  = test[features]
    y_test  = test[target]

    # Train
    pipeline.fit(X_train, y_train)

    # Predict and calculate metrics
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Normalized Mean Absolute Error
    mean_y = y_test.mean()         # promedio del valor real
    nmae = mae / mean_y            # MAE normalizado

    # Save pipeline y metrics
    results_dict[region] = {
        "pipeline": pipeline,
        "mae": mae,
        "rmse": rmse,
        "mean_y": mean_y,
        "nmae": nmae

    }

# Performance
for region, res in results_dict.items():
    print(f"{region}: MAE={res['mae']:.2f}, RMSE={res['rmse']:.2f}, NMAE={res['nmae']:.2f}")


ES11: MAE=3051011.50, RMSE=3908827.31, NMAE=0.39
ES12: MAE=1123428.50, RMSE=1442828.73, NMAE=0.24
ES13: MAE=1201427.00, RMSE=1662070.03, NMAE=0.28
ES21: MAE=2402713.50, RMSE=3180425.55, NMAE=0.42
ES22: MAE=864244.50, RMSE=1112850.20, NMAE=0.36
ES23: MAE=501026.75, RMSE=687523.07, NMAE=0.44
ES24: MAE=2016117.50, RMSE=2684489.90, NMAE=0.33


NMAE	Interpretación
- 0.00 – 0.05	Excelente (error 0–5% del valor real)
- 0.05 – 0.10	Muy bueno
- 0.10 – 0.20	Aceptable para datos sociales/macroeconómicos
- ">" 0.20	Bajo desempeño
- ">" 1.0	Muy malo (error > 100% del valor real)

In [88]:
# Dict with performance metrics
data = []
for region, res in results_dict.items():
    data.append({
        "region": region,
        "mae": res["mae"],
        "rmse": res["rmse"],
        "nmae": res['nmae']

    })

# Convert to DataFrame
df_results = pd.DataFrame(data)


df_results


,region,mae,rmse,nmae
0,ES11,3051011.50,3.908827e+06,0.387869
1,ES12,1123428.50,1.442829e+06,0.241222
2,ES13,1201427.00,1.662070e+06,0.277181
3,ES21,2402713.50,3.180426e+06,0.419514
4,ES22,864244.50,1.112850e+06,0.358182
5,ES23,501026.75,6.875231e+05,0.441459
6,ES24,2016117.50,2.684490e+06,0.325360
